# Advanced Sea Level Rise Analysis

This notebook provides a comprehensive analysis of global sea level rise using EPA data from 1880 to 2013.

## Features:
- Multiple regression models (linear, polynomial)
- Statistical analysis and trend detection
- Future projections with uncertainty quantification
- Interactive visualizations
- Model performance comparison

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sea_level_predictor import SeaLevelPredictor, create_advanced_analysis
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Data Loading and Exploration

In [ ]:
# Initialize the predictor and load data
predictor = SeaLevelPredictor()
data = predictor.preprocess_data()

print(f"Dataset shape: {data.shape}")
print(f"Date range: {data['Year'].min()} - {data['Year'].max()}")
print(f"\nFirst few rows:")
data.head()

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print(data['CSIRO Adjusted Sea Level'].describe())

# Check for missing values
print(f"\nMissing values: {data.isnull().sum().sum()}")

# Data quality check
print(f"\nData quality:")
print(f"- Years with data: {len(data)}")
print(f"- Consecutive years: {data['Year'].max() - data['Year'].min() + 1}")
print(f"- Data completeness: {len(data) / (data['Year'].max() - data['Year'].min() + 1) * 100:.1f}%")

## 2. Exploratory Data Analysis

In [ ]:
# Create exploratory plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Time series plot
axes[0, 0].plot(data['Year'], data['CSIRO Adjusted Sea Level'], linewidth=1.5, alpha=0.8)
axes[0, 0].set_title('Sea Level Over Time')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Sea Level (inches)')
axes[0, 0].grid(True, alpha=0.3)

# Distribution of sea levels
axes[0, 1].hist(data['CSIRO Adjusted Sea Level'], bins=30, alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Distribution of Sea Levels')
axes[0, 1].set_xlabel('Sea Level (inches)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Year-over-year changes
data['Annual_Change'] = data['CSIRO Adjusted Sea Level'].diff()
axes[1, 0].plot(data['Year'][1:], data['Annual_Change'][1:], alpha=0.7)
axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Annual Changes in Sea Level')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Annual Change (inches)')
axes[1, 0].grid(True, alpha=0.3)

# Decadal trends
decade_stats = data.groupby('Decade')['CSIRO Adjusted Sea Level'].agg(['mean', 'std']).reset_index()
axes[1, 1].bar(decade_stats['Decade'], decade_stats['mean'], 
               yerr=decade_stats['std'], alpha=0.7, capsize=5)
axes[1, 1].set_title('Decadal Average Sea Levels')
axes[1, 1].set_xlabel('Decade')
axes[1, 1].set_ylabel('Average Sea Level (inches)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Model Training and Comparison

In [ ]:
# Fit multiple models
print("Training models...")

# Linear models
predictor.fit_linear_model()  # All data
predictor.fit_linear_model(2000)  # From 2000 onwards
predictor.fit_linear_model(1950)  # From 1950 onwards

# Polynomial models
predictor.fit_polynomial_model(2)  # Quadratic
predictor.fit_polynomial_model(3)  # Cubic
predictor.fit_polynomial_model(2, 2000)  # Quadratic from 2000

print("Models trained successfully!")

# Display model performance
print("\nModel Performance:")
print("-" * 60)
print(f"{'Model':<25} {'R²':<10} {'RMSE':<10} {'Start Year':<10}")
print("-" * 60)

for key, model_info in predictor.models.items():
    print(f"{key.replace('_', ' ').title():<25} {model_info['r2']:<10.4f} {model_info['rmse']:<10.3f} {model_info['start_year']:<10}")

## 4. Future Projections

In [ ]:
# Generate predictions
predictor.predict_future(2100)

# Visualize predictions
plt.figure(figsize=(14, 8))

# Plot historical data
plt.scatter(data['Year'], data['CSIRO Adjusted Sea Level'], 
           alpha=0.6, s=20, color='blue', label='Historical Data')

# Plot predictions
colors = ['red', 'green', 'orange', 'purple', 'brown']
for i, (key, pred_info) in enumerate(predictor.predictions.items()):
    if i < len(colors):
        plt.plot(pred_info['years'], pred_info['predictions'], 
                color=colors[i], linewidth=2, linestyle='--',
                label=f"Prediction: {key.replace('_', ' ').title()}")

plt.axvline(x=data['Year'].max(), color='black', linestyle=':', alpha=0.7, 
           label='End of Historical Data')

plt.xlabel('Year', fontsize=12)
plt.ylabel('Sea Level (inches)', fontsize=12)
plt.title('Sea Level Projections to 2100', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print specific predictions
prediction_years = [2030, 2050, 2070, 2100]
print("\nFuture Predictions:")
print("-" * 80)
print(f"{'Year':<8} {'Linear (All)':<15} {'Linear (2000+)':<15} {'Polynomial':<15} {'Difference':<15}")
print("-" * 80)

for year in prediction_years:
    predictions = {}
    for key, pred_info in predictor.predictions.items():
        year_idx = np.where(pred_info['years'] == year)[0]
        if len(year_idx) > 0:
            predictions[key] = pred_info['predictions'][year_idx[0]]
    
    if 'linear_all' in predictions and 'linear_2000' in predictions:
        diff = predictions['linear_2000'] - predictions['linear_all']
        poly_pred = predictions.get('poly_2_all', 0)
        print(f"{year:<8} {predictions['linear_all']:<15.2f} {predictions['linear_2000']:<15.2f} {poly_pred:<15.2f} {diff:<15.2f}")

## 5. Statistical Analysis

In [ ]:
# Calculate comprehensive statistics
stats = predictor.calculate_statistics()

print("Comprehensive Statistical Analysis:")
print("=" * 50)
print(f"Total rise since 1880: {stats['total_rise']:.2f} inches ({stats['total_rise']*2.54:.1f} cm)")
print(f"Average annual rise: {stats['avg_annual_rise']:.3f} inches/year ({stats['avg_annual_rise']*2.54:.2f} cm/year)")

if stats['acceleration']:
    print(f"Acceleration in rise rate: {stats['acceleration']:.4f} inches/year²")
    print(f"Recent trend (2000+): {stats['recent_trend']:.3f} inches/year")

# Additional analysis
print("\nAdditional Insights:")
print("-" * 30)

# Rate changes by period
periods = {
    '1880-1920': (1880, 1920),
    '1920-1960': (1920, 1960),
    '1960-2000': (1960, 2000),
    '2000-2013': (2000, 2013)
}

for period, (start, end) in periods.items():
    period_data = data[(data['Year'] >= start) & (data['Year'] <= end)]
    if len(period_data) > 1:
        from scipy.stats import linregress
        slope, _, r_value, _, _ = linregress(period_data['Year'], period_data['CSIRO Adjusted Sea Level'])
        print(f"{period}: {slope:.3f} inches/year (R² = {r_value**2:.3f})")

# Variability analysis
print(f"\nVariability Analysis:")
print(f"Standard deviation: {data['CSIRO Adjusted Sea Level'].std():.3f} inches")
print(f"Coefficient of variation: {data['CSIRO Adjusted Sea Level'].std() / data['CSIRO Adjusted Sea Level'].mean() * 100:.1f}%")

# Trend significance
from scipy.stats import linregress
slope, intercept, r_value, p_value, std_err = linregress(data['Year'], data['CSIRO Adjusted Sea Level'])
print(f"\nTrend Significance:")
print(f"Overall trend p-value: {p_value:.2e}")
print(f"Trend is {'significant' if p_value < 0.05 else 'not significant'} at α = 0.05")

## 6. Advanced Visualizations

In [ ]:
# Run the complete advanced analysis
predictor_advanced, stats_advanced = create_advanced_analysis()

## 7. Model Residuals Analysis

In [ ]:
# Analyze model residuals
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Residuals Analysis', fontsize=16, fontweight='bold')

# Linear model residuals (all data)
linear_model = predictor.models['linear_all']['model']
X = data['Year'].values.reshape(-1, 1)
y_pred = linear_model.predict(X)
residuals = data['CSIRO Adjusted Sea Level'].values - y_pred

# Residuals vs fitted
axes[0, 0].scatter(y_pred, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (Linear Model)')
axes[0, 0].grid(True, alpha=0.3)

# Residuals over time
axes[0, 1].plot(data['Year'], residuals, alpha=0.7)
axes[0, 1].axhline(y=0, color='red', linestyle='--')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals Over Time')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normal Distribution)')
axes[1, 0].grid(True, alpha=0.3)

# Histogram of residuals
axes[1, 1].hist(residuals, bins=20, alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Residuals')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical tests on residuals
from scipy.stats import shapiro, jarque_bera

print("Residuals Analysis:")
print("-" * 30)
print(f"Mean residual: {np.mean(residuals):.6f}")
print(f"Std residual: {np.std(residuals):.3f}")

# Normality tests
shapiro_stat, shapiro_p = shapiro(residuals)
jb_stat, jb_p = jarque_bera(residuals)

print(f"\nNormality Tests:")
print(f"Shapiro-Wilk p-value: {shapiro_p:.4f}")
print(f"Jarque-Bera p-value: {jb_p:.4f}")
print(f"Residuals are {'normally distributed' if min(shapiro_p, jb_p) > 0.05 else 'not normally distributed'} at α = 0.05")

## 8. Conclusions and Recommendations

### Key Findings:

1. **Accelerating Trend**: Sea level rise has accelerated significantly since 2000
2. **Model Performance**: Polynomial models capture non-linear trends better than linear models
3. **Future Projections**: Different models show varying predictions, highlighting uncertainty
4. **Statistical Significance**: The overall upward trend is highly statistically significant

### Recommendations:

1. **Use ensemble methods** combining multiple models for more robust predictions
2. **Include uncertainty quantification** in all future projections
3. **Regular model updates** as new data becomes available
4. **Consider external factors** like temperature, ice sheet dynamics, and thermal expansion

### Limitations:

1. **Data period**: Limited to 1880-2013, missing recent accelerated changes
2. **Model assumptions**: Linear and polynomial models may not capture complex climate dynamics
3. **External factors**: Does not account for climate feedback loops or tipping points
4. **Regional variations**: Global average may mask significant regional differences